# 🌙 Low-Light Image Enhancement bằng CNN nhẹ — DarkIR-Edge (C=24)
**Môn học:** Xử Lý Ảnh và Thị Giác Máy Tính (CS2203) — Master of Science in Computer Science 2026 (UIT - ĐHQG TP.HCM)  
**Môi trường:** Google Colab (GPU NVIDIA Tesla T4 / L4)

---

### 🧭 Cấu trúc Notebook (Tách bạch rõ ràng Mô hình Chính vs. Mô hình Đối chuẩn Tùy chọn)
1. **Cell 1 – 6 (Setup & Dữ liệu):** Khởi tạo, giải nén `llie_data_clean.zip` vào ổ SSD cục bộ `/content/datasets`, nạp `split_manifest.json` cố định (`1.200 Train` + `100 Val` + `150 Test`) và bộ nạp dữ liệu xác định (`MASTER_SEED = 42`).
2. **Cell 8 (**TÂM ĐIỂM ĐỒ ÁN** — Kiến trúc đề xuất `DarkIR-Edge`):** Đặt riêng biệt trong một cell độc lập, chú thích tường minh từng khối của **DarkIR (CVPR 2025)** (`FreMLP`, `Di-SpAM`, `SCA`, `SimpleGate`, `Side Supervision`) cùng **2 cải tiến vệ tinh**:
   - **Cải tiến vệ tinh 1 (Lấy cảm hứng từ HVI-CIDNet CVPR 2025 & Lý thuyết Retinex):** Trích xuất kênh độ rọi cực đại **Max-RGB Illumination Prior ($I_{\max} = \max(R,G,B)$)** ghép vào đầu vào tạo thành tensor 4 kênh (`Intro Conv 4 -> C`), kết hợp hàm mất mát **Cosine Color Loss ($\mathcal{L}_{\text{color}}$)** ở Cell 11 để triệt tiêu lệch màu sRGB mà không làm tăng tham số suy luận.
   - **Cải tiến vệ tinh 2 (Kế thừa từ Edge-LLIE CVPRW 2024):** Cổng lọc nhiễu dư **Residual Gated Skip Connection** (`GatedSkip`) trên 3 tầng nối tắt để chặn nhiễu hạt cảm biến smartphone tràn sang Decoder.
3. **Cell 9 (Baseline Đối chuẩn Tùy chọn — `Standard U-Net`):** Tách riêng sang cell độc lập để chỉ dùng khi cần đối chuẩn.
4. **Cell 14 (**ƯU TIÊN SỐ 1** — Huấn luyện `DarkIR-Edge`):** Cell chạy riêng cho mô hình chính của đồ án.
5. **Cell 15 (Tùy chọn — Huấn luyện `Standard U-Net`):** Tách riêng hoàn toàn, có cờ `RUN_UNET_BASELINE = True/False` để bạn chủ động bật/tắt tùy thời gian Colab.
6. **Cell 16 – 20 (Đánh giá, Bảng kết quả, Biểu đồ & Hiệu năng phần cứng).**

In [ ]:
# ===== CELL 1: Imports, Hạt giống Cố định (MASTER_SEED) & Cấu hình Toàn cục =====
import os, sys, re, json, glob, time, math, random, hashlib, copy, zipfile, shutil, subprocess
from pathlib import Path
import numpy as np, cv2, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

IN_COLAB = 'google.colab' in sys.modules
TEST_MODE = os.environ.get('LLIE_TEST') == '1'          # Chỉ bật khi chạy kiểm thử tự động ngoài Colab
BASE = '/content' if IN_COLAB else os.environ.get('LLIE_BASE', './llie_run')
DRIVE = '/content/drive/MyDrive/llie_project' if IN_COLAB else f'{BASE}/drive'

# -------------------------------------------------------------------------
# QUẢN LÝ HẠT GIỐNG NGẪU NHIÊN CỐ ĐỊNH (DETERMINISTIC SEEDING)
# - MASTER_SEED = 42 được khóa cứng ngay từ đầu để mọi thao tác chia dữ liệu,
#   khởi tạo trọng số và cắt patch (Random Crop) ở các lần chạy lại cùng seed
#   đều tái lập chính xác 100% (không có bất kỳ sự ngẫu nhiên trôi dạt nào).
# -------------------------------------------------------------------------
MASTER_SEED = 42

def set_seed(seed=MASTER_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(MASTER_SEED)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEV.type == 'cuda'

# -------------------------------------------------------------------------
# CẤU HÌNH PHIÊN BẢN THỰC NGHIỆM & QUẢN LÝ VERSION TRÊN GOOGLE DRIVE
# - Cho phép train đi train lại nhiều lần với các siêu tham số, loss weights
#   hoặc thử nghiệm khác nhau mà KHÔNG bị ghi đè hay mất mát dữ liệu cũ.
# - Mỗi version lưu riêng biệt tại:
#     /content/drive/MyDrive/llie_project/experiments/{EXP_VERSION}/
#       ├── run_config.json    (siêu tham số, cấu hình chi tiết của version)
#       ├── checkpoints/       (chứa best.pt, last.pt, final.pt, history.csv)
#       └── results/           (chứa csv, biểu đồ png, ảnh phục chế mẫu)
# - Đồng thời tự động cập nhật bản tóm tắt mới nhất vào:
#     /content/drive/MyDrive/llie_project/results_latest/
# -------------------------------------------------------------------------
EXP_VERSION = 'v1'             # Đổi tên khi chạy lần mới: 'v1', 'v2', 'v2_lr1e-4', 'v3_loss03', v.v.
FORCE_RETRAIN = False          # Đặt True nếu muốn ép train lại từ đầu trong cùng version (bỏ qua final.pt)

EXP_DIR    = f'{DRIVE}/experiments/{EXP_VERSION}'
CKPT_DIR   = f'{EXP_DIR}/checkpoints'
RESULT_DIR = f'{EXP_DIR}/results'
LATEST_DIR = f'{DRIVE}/results_latest'

CFG = dict(
    version    = EXP_VERSION,
    force_retrain = FORCE_RETRAIN,
    zip_path   = f'{DRIVE}/llie_data_clean.zip',       # File zip dữ liệu trên Google Drive
    data_root  = f'{BASE}/datasets',                   # Thư mục giải nén trên ổ SSD cục bộ của Colab
    exp_dir    = EXP_DIR,                              # Thư mục riêng của version trên Google Drive
    ckpt_dir   = CKPT_DIR,                             # Thư mục lưu checkpoint theo version
    result_dir = RESULT_DIR,                           # Thư mục lưu kết quả csv/png theo version
    latest_dir = LATEST_DIR,                           # Thư mục đồng bộ kết quả mới nhất trên Drive
    manifest   = f'{DRIVE}/split_manifest.json',       # File phân chia dữ liệu dùng chung bất biến
    # --- Siêu tham số huấn luyện cố định (khớp 100% Tài liệu 02 & 03) ---
    epochs=100, batch_size=8, patch=256, repeat=2,     # 1200 ảnh * 2 / 8 = 300 steps/epoch -> 30,000 steps
    lr=2e-4, lr_min=1e-6, weight_decay=1e-4, betas=(0.9, 0.999),
    val_interval=5, ema_decay=0.999, grad_clip=1.0, num_workers=2,
    loss_w=dict(side=0.5, ssim=0.2, color=0.1),
    # --- Phân bổ dữ liệu (1,200 Train : 100 Val : 150 Test = 1,450 cặp) ---
    n_train=dict(lolv2=639, huawei=561),
    n_val=dict(lolv2=50, huawei=50),
    huawei_gap=10,
    # --- Hạt giống huấn luyện cho từng mô hình (tách riêng rõ ràng) ---
    darkir_edge_seeds=[42, 123],                       # Ưu tiên chạy DarkIR-Edge (seed chính 42, seed kiểm chứng 123)
    unet_seeds=[42],                                   # Baseline đối chuẩn (chỉ chạy ở Cell 15 khi bật RUN_UNET_BASELINE)
    unet_base=32,                                      # base=32 -> 1,928,483 params (khớp 1,924,038 của DarkIR-Edge C=24)
    # --- Cờ Early Stopping (mặc định False = tạm khóa theo thiết kế để chạy trọn vẹn 100 epochs CosineAnnealing) ---
    use_early_stopping=False,
    early_stop_patience=20,                            # Số lần kiểm tra val liên tiếp không tăng nếu bật use_early_stopping=True
    max_steps_per_epoch=None,
)

if TEST_MODE:
    CFG.update(
        epochs=2, batch_size=2, patch=64, repeat=2, val_interval=1, num_workers=0,
        n_train=dict(lolv2=6, huawei=6), n_val=dict(lolv2=2, huawei=2), huawei_gap=1,
        darkir_edge_seeds=[42], unet_seeds=[42], max_steps_per_epoch=3
    )

os.makedirs(CFG['data_root'], exist_ok=True)
if not IN_COLAB:
    for d in (DRIVE, CFG['exp_dir'], CFG['ckpt_dir'], CFG['result_dir'], CFG['latest_dir']):
        os.makedirs(d, exist_ok=True)

try:
    import lpips
    LPIPS_OK = True
except Exception:
    if IN_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips'])
    try:
        import lpips; LPIPS_OK = True
    except Exception as e:
        LPIPS_OK = False; print('WARN: Không có thư viện lpips -> tự động bỏ qua chỉ số LPIPS.', e)

print(f'PyTorch {torch.__version__} | Device: {DEV} | AMP: {USE_AMP} | MASTER_SEED: {MASTER_SEED}')
print(f'Phiên bản thực nghiệm (Version): "{CFG["version"]}"')
print(f'Thư mục đích Google Drive: {CFG["exp_dir"]}')

In [ ]:
# ===== CELL 2: Kết nối Google Drive & Giải nén dữ liệu vào ổ SSD cục bộ (/content/datasets) =====
if IN_COLAB:
    # Nếu trước đó có thư mục cục bộ /content/drive chưa mount gây xung đột thì dọn sạch
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    from google.colab import drive
    drive.mount('/content/drive')
    # Khởi tạo đầy đủ cấu trúc thư mục trên Google Drive (bao gồm phân tầng version)
    for d in (DRIVE, CFG['exp_dir'], CFG['ckpt_dir'], CFG['result_dir'], CFG['latest_dir']):
        os.makedirs(d, exist_ok=True)
    # Lưu bản ghi cấu hình run_config.json vào thư mục của version này trên Drive để phục vụ truy vết khoa học
    cfg_record = {k: (str(v) if isinstance(v, (torch.device, Path)) else v) for k, v in CFG.items()}
    with open(f"{CFG['exp_dir']}/run_config.json", 'w', encoding='utf-8') as f_cfg:
        json.dump(cfg_record, f_cfg, indent=2, ensure_ascii=False)
    print(f'✔ Đã mount Google Drive! Thư mục lưu kết quả phiên bản "{CFG["version"]}":')
    print(f'   -> Checkpoints: {CFG["ckpt_dir"]}')
    print(f'   -> Results:     {CFG["result_dir"]}')

    if not os.listdir(CFG['data_root']):
        assert os.path.exists(CFG['zip_path']), f"Không tìm thấy file zip tại: {CFG['zip_path']}"
        print('Đang copy và giải nén', CFG['zip_path'], 'vào SSD cục bộ...')
        shutil.copy(CFG['zip_path'], '/content/llie_data.zip')
        with zipfile.ZipFile('/content/llie_data.zip') as z:
            z.extractall(CFG['data_root'])
        # Nếu trong zip đã đóng gói sẵn split_manifest.json chuẩn từ máy local thì copy sang DRIVE
        bundled_man = Path(CFG['data_root']) / 'split_manifest.json'
        if bundled_man.exists() and not os.path.exists(CFG['manifest']):
            shutil.copy(str(bundled_man), CFG['manifest'])
            print('Đã nạp split_manifest.json đóng gói sẵn từ file zip vào Google Drive.')
    print('Dữ liệu sẵn sàng tại:', CFG['data_root'], os.listdir(CFG['data_root']))
elif TEST_MODE:
    fake_specs = {
        'lolv2_real_train': 10, 'lolv2_real_test': 4,
        'lsrw_huawei_train': 10, 'lsrw_huawei_test': 3,
        'lsrw_nikon_test': 3,
    }
    for k, n in fake_specs.items():
        ld = Path(CFG['data_root']) / 'fake' / k / 'low'
        hd = Path(CFG['data_root']) / 'fake' / k / 'high'
        ld.mkdir(parents=True, exist_ok=True); hd.mkdir(parents=True, exist_ok=True)
        for i in range(1, n + 1):
            H, W = (70, 92) if i % 2 == 0 else (68, 84)
            hi_img = np.random.randint(80, 240, (H, W, 3), dtype=np.uint8)
            lo_img = np.clip((hi_img.astype(np.float32) / 255.0) ** 2.5 * 255.0 + np.random.randn(H, W, 3) * 5, 0, 255).astype(np.uint8)
            cv2.imwrite(str(ld / f'low{i:05d}.png'), lo_img)
            cv2.imwrite(str(hd / f'normal{i:05d}.png'), hi_img)
    print('Đã khởi tạo dữ liệu giả lập cho TEST_MODE.')
else:
    print('Chạy cục bộ ngoài Colab. Thư mục dữ liệu:', CFG['data_root'])

In [ ]:
# ===== CELL 3: Kiểm tra cấu trúc cây thư mục thực tế sau khi giải nén =====
def show_tree(root, depth=4, max_items=5):
    root = Path(root)
    if not root.exists():
        print('Thư mục chưa tồn tại:', root); return
    def rec(p, d):
        if d > depth: return
        kids = sorted(p.iterdir())
        dirs = [k for k in kids if k.is_dir()]
        files = [k for k in kids if k.is_file()]
        for k in dirs:
            print('  ' * d + f'[D] {k.name}/')
            rec(k, d + 1)
        if files:
            sample = ', '.join(f.name for f in files[:max_items])
            print('  ' * d + f'    ({len(files)} files) vd: {sample}')
    rec(root, 0)

show_tree(CFG['data_root'])

### 🔒 Phân chia Dữ liệu Cố định (`MASTER_SEED = 42`) & Bảo đảm Zero Data Leakage
- Trong Cell 5 dưới đây:
  - Bộ **LOLv2-Real** (`689` cặp Train): Lấy cố định **50 cặp cuối** làm Validation, **639 cặp đầu** làm Train (tránh rò rỉ các ảnh chụp liên tiếp cùng một căn phòng).
  - Bộ **LSRW-Huawei** (`1.000` cặp Train): Lấy cố định **50 cặp cuối** làm Validation, chừa vùng đệm `gap = 10` ảnh, và dùng đối tượng `random.Random(MASTER_SEED)` với `MASTER_SEED = 42` để rút cố định **561 cặp** làm Train.
  - Mọi đường dẫn trong `split_manifest.json` được lưu ở dạng **đường dẫn tương đối** so với `CFG['data_root']` kèm mã băm **SHA-256**, đảm bảo tạo ở máy local hay trên Google Colab đều ra **cùng một danh sách ảnh giống hệt nhau 100%**.

In [ ]:
# ===== CELL 5: Ghép cặp ảnh + Chia Train/Val/Test cố định (MASTER_SEED=42) + Lưu split_manifest.json =====
CANDIDATE_PATHS = {
    'lolv2_real_train': [
        ('LOL-v2/Real_captured/Train/Low', 'LOL-v2/Real_captured/Train/Normal'),
        ('Real_captured/Train/Low',        'Real_captured/Train/Normal'),
        ('LOLv2-Real/Train/Low',           'LOLv2-Real/Train/Normal'),
    ],
    'lolv2_real_test': [
        ('LOL-v2/Real_captured/Test/Low',  'LOL-v2/Real_captured/Test/Normal'),
        ('LOL-v2/Real_captured/test/Low',  'LOL-v2/Real_captured/test/Normal'),
        ('Real_captured/Test/Low',         'Real_captured/Test/Normal'),
        ('LOLv2-Real/Test/Low',            'LOLv2-Real/Test/Normal'),
    ],
    'lsrw_huawei_train': [
        ('LSRW/Train/Huawei/low',          'LSRW/Train/Huawei/high'),
        ('LSRW/Training data/Huawei/low',  'LSRW/Training data/Huawei/high'),
        ('data1/Training data/Huawei/low', 'data1/Training data/Huawei/high'),
        ('Train/Huawei/low',               'Train/Huawei/high'),
        ('LSRW/Huawei/Train/low',          'LSRW/Huawei/Train/high'),
    ],
    'lsrw_huawei_test': [
        ('LSRW/Eval/Huawei/low',           'LSRW/Eval/Huawei/high'),
        ('data1/Eval/Huawei/low',          'data1/Eval/Huawei/high'),
        ('LSRW_/Eval/Huawei/low',          'LSRW_/Eval/Huawei/high'),
        ('Eval/Huawei/low',                'Eval/Huawei/high'),
        ('LSRW/Huawei/Test/low',           'LSRW/Huawei/Test/high'),
    ],
    'lsrw_nikon_test': [
        ('LSRW/Eval/Nikon/low',            'LSRW/Eval/Nikon/high'),
        ('data1/Eval/Nikon/low',           'data1/Eval/Nikon/high'),
        ('LSRW_/Eval/Nikon/low',           'LSRW_/Eval/Nikon/high'),
        ('Eval/Nikon/low',                 'Eval/Nikon/high'),
        ('LSRW/Nikon/Test/low',            'LSRW/Nikon/Test/high'),
    ],
}

def resolve_sources(data_root):
    if TEST_MODE:
        return {k: (f'fake/{k}/low', f'fake/{k}/high') for k in CANDIDATE_PATHS}
    resolved = {}
    root = Path(data_root)
    for key, cands in CANDIDATE_PATHS.items():
        found = None
        for lo_rel, hi_rel in cands:
            if (root / lo_rel).is_dir() and (root / hi_rel).is_dir():
                found = (lo_rel, hi_rel); break
        assert found is not None, f"Không tìm thấy thư mục cho '{key}' trong {root}. Hãy kiểm tra Cell 3!"
        resolved[key] = found
    return resolved

SOURCES = resolve_sources(CFG['data_root'])
print('Đường dẫn dữ liệu đã nhận diện:', json.dumps(SOURCES, indent=2, ensure_ascii=False))

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
def _nat(s): return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)]
def _key(stem): return re.sub(r'(?i)^(low|normal|high|gt)[_\-. ]*|[_\-. ]*(low|normal|high|gt)$', '', stem).strip('_-. ')

def make_pairs(low_rel, high_rel):
    root = Path(CFG['data_root'])
    ld, hd = root / low_rel, root / high_rel
    assert ld.is_dir() and hd.is_dir(), f'Không thấy thư mục: {ld} hoặc {hd}'
    low  = {_key(p.stem): p.relative_to(root).as_posix() for p in ld.iterdir() if p.suffix.lower() in IMG_EXT}
    high = {_key(p.stem): p.relative_to(root).as_posix() for p in hd.iterdir() if p.suffix.lower() in IMG_EXT}
    diff = set(low) ^ set(high)
    assert not diff, f'{len(diff)} ảnh không khớp cặp giữa {ld} và {hd}, vd: {sorted(diff, key=_nat)[:5]}'
    return [(low[k], high[k]) for k in sorted(low, key=_nat)]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def to_abs_pairs(rel_pairs):
    root = Path(CFG['data_root'])
    res = []
    for l, h in rel_pairs:
        lp = Path(l) if Path(l).is_absolute() and Path(l).exists() else (root / l)
        hp = Path(h) if Path(h).is_absolute() and Path(h).exists() else (root / h)
        res.append((str(lp), str(hp)))
    return res

def build_manifest():
    root = Path(CFG['data_root'])
    nv, nt, gap = CFG['n_val'], CFG['n_train'], CFG['huawei_gap']
    lol = make_pairs(*SOURCES['lolv2_real_train'])
    hw  = make_pairs(*SOURCES['lsrw_huawei_train'])

    # 1. LOLv2-Real: 50 cặp cuối làm Val, 639 cặp đầu làm Train
    lol_val, lol_pool = lol[-nv['lolv2']:], lol[:-nv['lolv2']]
    if len(lol_pool) <= nt['lolv2']:
        lol_train = lol_pool
    else:
        idx = sorted(random.Random(MASTER_SEED).sample(range(len(lol_pool)), nt['lolv2']))
        lol_train = [lol_pool[i] for i in idx]

    # 2. LSRW-Huawei: 50 cặp cuối làm Val, vùng đệm gap=10, lấy mẫu cố định bằng MASTER_SEED=42
    hw_val, hw_pool = hw[-nv['huawei']:], hw[:len(hw) - nv['huawei'] - gap]
    idx = sorted(random.Random(MASTER_SEED).sample(range(len(hw_pool)), nt['huawei']))
    hw_train = [hw_pool[i] for i in idx]

    splits_rel = {
        'train': {'lolv2': lol_train, 'huawei': hw_train},
        'val':   {'lolv2': lol_val,   'huawei': hw_val},
        'test':  {
            'lolv2':  make_pairs(*SOURCES['lolv2_real_test']),
            'huawei': make_pairs(*SOURCES['lsrw_huawei_test']),
            'nikon':  make_pairs(*SOURCES['lsrw_nikon_test']),
        },
    }
    sets = {(s, d): {p[0] for p in v} for s, dd in splits_rel.items() for d, v in dd.items()}
    keys = list(sets)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            assert not (sets[keys[i]] & sets[keys[j]]), f'PHÁT HIỆN RÒ RỈ DỮ LIỆU giữa {keys[i]} và {keys[j]}'

    man = {
        s: {d: [[l, h, sha256(root / l), sha256(root / h)] for l, h in v] for d, v in dd.items()}
        for s, dd in splits_rel.items()
    }
    json.dump(man, open(CFG['manifest'], 'w'), indent=1)
    return {s: {d: to_abs_pairs(v) for d, v in dd.items()} for s, dd in splits_rel.items()}

if os.path.exists(CFG['manifest']) and not TEST_MODE:
    man = json.load(open(CFG['manifest']))
    SPLITS = {s: {d: to_abs_pairs([(x[0], x[1]) for x in v]) for d, v in dd.items()} for s, dd in man.items()}
    print('Đã nạp manifest cố định có sẵn:', CFG['manifest'])
else:
    SPLITS = build_manifest()
    print('Đã tạo mới manifest cố định kèm SHA-256:', CFG['manifest'])

for s, dd in SPLITS.items():
    print(f'{s:5s}', {d: len(v) for d, v in dd.items()}, '-> Tổng:', sum(len(v) for v in dd.values()))

In [ ]:
# ===== CELL 6: Dataset & DataLoader (Cắt Patch 256x256 xác định theo Seed, KHÔNG Resize/ColorJitter) =====
def imread_rgb(p):
    im = cv2.imread(p, cv2.IMREAD_COLOR)
    if im is None: raise IOError(f'Không đọc được file ảnh: {p}')
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def to_tensor(x):
    return torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1).float().div_(255.0)

class PairDataset(Dataset):
    def __init__(self, pairs, patch=256, repeat=1, train=True):
        self.pairs, self.patch, self.repeat, self.train = list(pairs), patch, repeat, train
        self.cache, nbytes = [], 0
        for lp, hp in self.pairs:
            lo, hi = imread_rgb(lp), imread_rgb(hp)
            assert lo.shape == hi.shape, f'Kích thước low/high không khớp: {lp} {lo.shape} vs {hi.shape}'
            self.cache.append((lo, hi))
            nbytes += lo.nbytes + hi.nbytes
        self.gb = nbytes / 1e9

    def __len__(self):
        return len(self.pairs) * (self.repeat if self.train else 1)

    def __getitem__(self, i):
        lo, hi = self.cache[i % len(self.pairs)]
        if self.train:
            H, W, _ = lo.shape; p = self.patch
            if H < p or W < p:
                ph, pw = max(0, p - H), max(0, p - W)
                lo = np.pad(lo, ((0, ph), (0, pw), (0, 0)), 'reflect')
                hi = np.pad(hi, ((0, ph), (0, pw), (0, 0)), 'reflect')
                H, W, _ = lo.shape
            # Các hàm torch.randint / torch.rand ở đây chịu sự điều khiển trực tiếp của set_seed(seed)
            # và worker_init_fn cố định ở DataLoader bên dưới -> đảm bảo tái lập 100% cùng tọa độ crop!
            y = int(torch.randint(0, H - p + 1, (1,)))
            x = int(torch.randint(0, W - p + 1, (1,)))
            lo, hi = lo[y:y + p, x:x + p], hi[y:y + p, x:x + p]
            if torch.rand(1) < 0.5: lo, hi = lo[:, ::-1], hi[:, ::-1]
            if torch.rand(1) < 0.5: lo, hi = lo[::-1], hi[::-1]
        return to_tensor(lo), to_tensor(hi)

def flat(split):
    return [p for d in SPLITS[split].values() for p in d]

train_ds = PairDataset(flat('train'), CFG['patch'], CFG['repeat'], train=True)
val_ds   = PairDataset(flat('val'), train=False)
print(f'train_ds: {len(train_ds)} patches/epoch ({len(train_ds.pairs)} ảnh × repeat {CFG["repeat"]}), RAM cache ≈ {train_ds.gb:.2f} GB | val_ds: {len(val_ds)} ảnh')

def _seed_worker(worker_id):
    # Đảm bảo mỗi worker đa luồng của PyTorch đều nhận seed cố định từ generator chính
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_train_loader(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        train_ds, batch_size=CFG['batch_size'], shuffle=True, drop_last=True,
        generator=g, worker_init_fn=_seed_worker,
        num_workers=CFG['num_workers'], pin_memory=DEV.type == 'cuda',
        persistent_workers=CFG['num_workers'] > 0
    )

steps_per_ep = len(train_ds) // CFG['batch_size']
print(f'Steps/epoch = {steps_per_ep} | Tổng ngân sách cố định = {CFG["epochs"] * steps_per_ep:,} steps')

## 🌟 Kiến trúc Đề xuất Duy nhất của Đồ án: `DarkIR-Edge (C=24)` (Cell 8)
Cell 8 dưới đây cài đặt **độc lập và trọn vẹn** mô hình đề xuất duy nhất **`DarkIR-Edge`** ($C=24$, `1.924.038` tham số), kết hợp xương sống **DarkIR (CVPR 2025)** với **2 cải tiến từ bài báo vệ tinh**:
1. **Vận dụng HVI-CIDNet (CVPR 2025) & Lý thuyết Retinex — Phối hợp Hiệp đồng (The Golden Synergy):**
   - **Tại đầu vào mô hình (`extract_imax_prior` + `self.intro` trong Cell 8):** Trích xuất bản đồ độ rọi **Max-RGB Illumination Prior** $I_{\max}(x, y) = \max_{c \in \{R,G,B\}} I_c(x, y)$ lấy cảm hứng từ ý tưởng phân tách cường độ sáng $I$ khỏi mặt phẳng sắc độ trong không gian HVI, ghép trực tiếp thành kênh thứ 4: $X_{\text{in}} = [I_R, I_G, I_B, I_{\max}] \in \mathbb{R}^{B \times 4 \times H \times W}$.
   - **Tại hàm mất mát (`CompositeLoss` trong Cell 11):** Kết hợp **Cosine Color Loss ($\mathcal{L}_{\text{color}}$)** kèm mặt nạ che vùng tối sâu (`mask = gt.amax(1) >= 5/255`) để phạt góc lệch vector màu RGB mà không phải dùng biến đổi lượng giác $\sin/\cos/\arctan$ nhạy cảm đạo hàm của nhánh HVI đầy đủ.
2. **Kế thừa Edge-LLIE (CVPRW 2024) — Cổng lọc nhiễu `GatedSkip`:**
   - Đặt trên cả 3 đường nối tắt từ Encoder sang Decoder: $F_{\text{gated}} = x_{\text{skip}} + \text{Conv}_{1\times1}(x_{\text{skip}}) \odot \sigma(\text{Conv}_{1\times1}(x_{\text{skip}}))$.

In [ ]:
# ===== CELL 8: KIẾN TRÚC MÔ HÌNH ĐỀ XUẤT CHÍNH CỦA ĐỒ ÁN — DarkIREdge (C=24) =====
# (Được tách riêng biệt hoàn toàn trong cell này để dễ đọc, dễ bảo vệ và đối chiếu 1:1 với Tài liệu 03)

class LayerNorm2d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        x32 = x.float()
        mu = x32.mean(1, keepdim=True)
        var = (x32 - mu).pow(2).mean(1, keepdim=True)
        y = (x32 - mu) / torch.sqrt(var + self.eps)
        return (y * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)).to(x.dtype)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class FreMLP(nn.Module):
    '''
    [DarkIR CVPR 2025 - Nhánh Tần Số Fourier]
    Điều biến biên độ phổ Fourier (Amplitude) để nâng sáng toàn cục, giữ nguyên pha (Phase).
    Nâng cấp kỹ thuật số học: Chạy ở FP32 và nhân trực tiếp với Phasor đơn vị (xf / mag),
    loại bỏ hoàn toàn hàm lượng giác atan2/cos/sin để chống bùng nổ đạo hàm dưới AMP FP16.
    '''
    def __init__(self, nc, expand=2, norm='backward', eps=1e-6):
        super().__init__()
        self.norm, self.eps = norm, eps
        self.process1 = nn.Sequential(
            nn.Conv2d(nc, expand * nc, 1, 1, 0),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(expand * nc, nc, 1, 1, 0)
        )

    def forward(self, x):
        H, W = x.shape[-2:]
        with torch.autocast(device_type=x.device.type, enabled=False):  # Bắt buộc tính cuFFT ở FP32
            xf = torch.fft.rfft2(x.float(), norm=self.norm)
            mag = torch.sqrt(xf.real ** 2 + xf.imag ** 2 + self.eps)
            mag_out = self.process1(mag)
            yf = xf * (mag_out / mag)                                   # xf / mag chính là Phasor đơn vị e^(j*P)
            y = torch.fft.irfft2(yf, s=(H, W), norm=self.norm)
        return y.to(x.dtype)

class Branch(nn.Module):
    '''Nhánh tích chập chiều sâu giãn nở (Dilated Depthwise Convolution) của DarkIR'''
    def __init__(self, c, DW_Expand, dilation=1):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.branch = nn.Sequential(
            nn.Conv2d(self.dw_channel, self.dw_channel, kernel_size=3, padding=dilation,
                      stride=1, groups=self.dw_channel, bias=True, dilation=dilation)
        )
    def forward(self, x):
        return self.branch(x)

class EBlock(nn.Module):
    '''Khối Encoder chuẩn DarkIR (DarkIR/archs/arch_model.py:141-203): Tích hợp SCA + SimpleGate + FreMLP'''
    def __init__(self, c, DW_Expand=2, dilations=(1,), extra_depth_wise=True, use_fft=True):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.extra_conv = nn.Conv2d(c, c, kernel_size=3, padding=1, stride=1, groups=c, bias=True, dilation=1) if extra_depth_wise else nn.Identity()
        self.conv1 = nn.Conv2d(c, self.dw_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.branches = nn.ModuleList([Branch(c, DW_Expand, dilation=d) for d in dilations])
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.dw_channel // 2, self.dw_channel // 2, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        )
        self.sg1 = SimpleGate()
        self.conv3 = nn.Conv2d(self.dw_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.freq = FreMLP(nc=c, expand=2) if use_fft else None
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(self.extra_conv(x))
        z = sum(branch(x) for branch in self.branches)
        z = self.sg1(z)
        x = self.sca(z) * z
        x = self.conv3(x)
        y = inp + self.beta * x
        # Bước 2: Điều biến nhân đặc trưng miền tần số (y * x_freq)
        x_step2 = self.norm2(y)
        if self.freq is not None:
            x_freq = self.freq(x_step2)
            x = y * x_freq
            x = y + x * self.gamma
        else:
            x = y
        return x

class DBlock(nn.Module):
    '''Khối Decoder chuẩn DarkIR (DarkIR/archs/arch_model.py:72-140): Cụm tích chập đa tỷ lệ Di-SpAM d=[1, 4, 9]'''
    def __init__(self, c, DW_Expand=2, FFN_Expand=2, dilations=(1, 4, 9), extra_depth_wise=True):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.conv1 = nn.Conv2d(c, self.dw_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.extra_conv = nn.Conv2d(self.dw_channel, self.dw_channel, kernel_size=3, padding=1, stride=1, groups=c, bias=True, dilation=1) if extra_depth_wise else nn.Identity()
        self.branches = nn.ModuleList([Branch(self.dw_channel, DW_Expand=1, dilation=d) for d in dilations])
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.dw_channel // 2, self.dw_channel // 2, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        )
        self.sg1 = SimpleGate()
        self.sg2 = SimpleGate()
        self.conv3 = nn.Conv2d(self.dw_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        ffn_channel = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.extra_conv(self.conv1(x))
        z = sum(branch(x) for branch in self.branches)
        z = self.sg1(z)
        x = self.sca(z) * z
        x = self.conv3(x)
        y = inp + self.beta * x
        # Bước 2: FFN khôi phục chi tiết không gian
        x = self.conv4(self.norm2(y))
        x = self.sg2(x)
        x = self.conv5(x)
        return y + x * self.gamma

# =========================================================================
# CẢI TIẾN VỆ TINH 1 (Lấy cảm hứng từ HVI-CIDNet CVPR 2025 & Lý thuyết Retinex):
# Trích xuất bản đồ độ rọi cực đại Max-RGB Illumination Prior: I_max = max(R, G, B)
# và ghép vào ảnh RGB đầu vào tạo thành Tensor 4 kênh [R, G, B, I_max].
# (Kết hợp cùng hàm Cosine Color Loss ở Cell 11 để bảo toàn sắc độ mà tốn 0 tham số ẩn).
# =========================================================================
def extract_imax_prior(rgb_tensor):
    '''Input: rgb_tensor (B, 3, H, W) -> Output: [R, G, B, I_max] (B, 4, H, W)'''
    i_max = rgb_tensor.amax(dim=1, keepdim=True)   # Kênh độ rọi I_max ∈ R^(B x 1 x H x W)
    return torch.cat([rgb_tensor, i_max], dim=1)

# =========================================================================
# CẢI TIẾN VỆ TINH 2 (Kế thừa từ Edge-LLIE CVPRW 2024):
# Cổng lọc nhiễu dư Residual Gated Skip Connection trên 3 tầng nối tắt Encoder -> Decoder
# F_gated = x_skip + Conv1x1(x_skip) * Sigmoid(Conv1x1(x_skip))
# =========================================================================
class GatedSkip(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.f = nn.Conv2d(c, c, 1)
        self.g = nn.Conv2d(c, c, 1)
    def forward(self, x):
        return x + self.f(x) * torch.sigmoid(self.g(x))

def pad8(x):
    '''Đệm phản xạ (Reflection Padding) lên bội số của 8 để hỗ trợ ảnh kích thước bất kỳ'''
    H, W = x.shape[-2:]
    ph, pw = (8 - H % 8) % 8, (8 - W % 8) % 8
    return F.pad(x, (0, pw, 0, ph), mode='reflect') if (ph or pw) else x

class DarkIREdge(nn.Module):
    '''
    KIẾN TRÚC ĐỀ XUẤT DUY NHẤT CỦA ĐỒ ÁN: DarkIR-Edge (width=24 -> 1,924,038 tham số)
    - Bảo toàn 100% cấu trúc bất đối xứng của DarkIR: Encoder [1, 2, 3] (EBlock + FreMLP),
      Bottleneck [2 EBlock + 2 DBlock + Side Supervision 8C->3], Decoder [3, 1, 1] (DBlock + Di-SpAM).
    - Tích hợp 2 cải tiến vệ tinh:
      (1) use_imax=True: Ghép kênh tiên nghiệm độ rọi Max-RGB (HVI/Retinex Inspiration) -> Intro Conv nhận 4 kênh.
      (2) use_gate=True: Cổng lọc nhiễu Residual Gated Skip Connection (Edge-LLIE) tại 3 tầng skip.
    '''
    def __init__(self, width=24, enc_blk_nums=(1, 2, 3), dec_blk_nums=(3, 1, 1),
                 middle_blk_num_enc=2, middle_blk_num_dec=2, dilations=(1, 4, 9),
                 extra_depth_wise=True, use_imax=True, use_gate=True, use_fft=True):
        super().__init__()
        self.use_imax = use_imax
        in_ch = 4 if use_imax else 3
        self.intro = nn.Conv2d(in_ch, width, kernel_size=3, padding=1, stride=1, bias=True)
        self.ending = nn.Conv2d(width, 3, kernel_size=3, padding=1, stride=1, bias=True)
        self.encoders, self.downs = nn.ModuleList(), nn.ModuleList()
        self.decoders, self.ups, self.gates = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()

        chan = width
        for num in enc_blk_nums:
            self.encoders.append(nn.Sequential(*[EBlock(chan, extra_depth_wise=extra_depth_wise, use_fft=use_fft) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, 2 * chan, 2, 2))
            chan *= 2

        self.middle_blks_enc = nn.Sequential(*[EBlock(chan, extra_depth_wise=extra_depth_wise, use_fft=use_fft) for _ in range(middle_blk_num_enc)])
        self.middle_blks_dec = nn.Sequential(*[DBlock(chan, dilations=dilations, extra_depth_wise=extra_depth_wise) for _ in range(middle_blk_num_dec)])
        self.side_out = nn.Conv2d(chan, 3, kernel_size=3, stride=1, padding=1)

        for num in dec_blk_nums:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan * 2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.gates.append(GatedSkip(chan) if use_gate else nn.Identity())
            self.decoders.append(nn.Sequential(*[DBlock(chan, dilations=dilations, extra_depth_wise=extra_depth_wise) for _ in range(num)]))

    def forward(self, inp, return_side=False):
        H, W = inp.shape[-2:]
        x_pad = pad8(inp)
        # Vận dụng Cải tiến Vệ tinh 1 (Max-RGB Illumination Prior từ cảm hứng HVI/Retinex)
        x_in = extract_imax_prior(x_pad) if self.use_imax else x_pad
        x = self.intro(x_in)

        skips = []
        for encoder, down in zip(self.encoders, self.downs):
            x = encoder(x)
            skips.append(x)
            x = down(x)

        x_light = self.middle_blks_enc(x)
        out_side = self.side_out(x_light) if return_side else None
        x = self.middle_blks_dec(x_light)
        x = x + x_light   # Kết nối tắt nội bộ tại Bottleneck (chuẩn DarkIR.py dòng 94)

        for decoder, up, gate, skip in zip(self.decoders, self.ups, self.gates, skips[::-1]):
            x = up(x)
            # Vận dụng Cải tiến Vệ tinh 2 (Residual Gated Skip Connection từ Edge-LLIE)
            x = x + gate(skip)
            x = decoder(x)

        x = self.ending(x) + x_pad
        out = x[..., :H, :W]
        if return_side and out_side is not None:
            out_side = out_side[..., :H // 8, :W // 8]
        return (out, out_side) if return_side else out

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'✔ Đã định nghĩa DarkIREdge (C=24) -> Tổng số tham số: {n_params(DarkIREdge(width=24)):,} params')

In [ ]:
# ===== CELL 9 (Tách riêng): Kiến trúc Baseline Đối chuẩn — Standard U-Net (base=32, ~1.93M params) =====
# Mô hình này chỉ dùng làm mốc đối chuẩn (Baseline) cùng quy mô tham số để so sánh với DarkIR-Edge.

def _cb(i, o, g=4):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True),
        nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True)
    )

class UNetBaseline(nn.Module):
    '''Standard U-Net tích chập thuần (Plain Conv2d), cùng độ sâu 3 tầng downsample & Global Residual'''
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.e1, self.e2, self.e3, self.mid = _cb(3, b), _cb(b, 2 * b), _cb(2 * b, 4 * b), _cb(4 * b, 8 * b)
        self.u3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2)
        self.u2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2)
        self.u1 = nn.ConvTranspose2d(2 * b, b, 2, 2)
        self.d3, self.d2, self.d1 = _cb(8 * b, 4 * b), _cb(4 * b, 2 * b), _cb(2 * b, b)
        self.out = nn.Conv2d(b, 3, 1)

    def forward(self, x, return_side=False):
        H, W = x.shape[-2:]
        x_pad = pad8(x)
        e1 = self.e1(x_pad)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        m  = self.mid(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        out = (self.out(d1) + x_pad)[..., :H, :W]
        return (out, None) if return_side else out

print(f'✔ Đã định nghĩa UNetBaseline (base={CFG["unet_base"]}) -> Tổng số tham số: {n_params(UNetBaseline(CFG["unet_base"])):,} params')

In [ ]:
# ===== CELL 10: Kiểm chứng số lượng tham số từng tầng & Smoke test kích thước ảnh lẻ =====
edge = DarkIREdge(width=24, use_imax=True, use_gate=True)
darkir_m_ref = DarkIREdge(width=32, use_imax=False, use_gate=False)
unet = UNetBaseline(base=CFG['unet_base'])

print(f'1. DarkIR-Edge (C=24, ĐỀ XUẤT CHÍNH): {n_params(edge):,} tham số (khớp chuẩn 1,924,038 trong File 03)')
print(f'2. Standard U-Net (base={CFG["unet_base"]}):          {n_params(unet):,} tham số (đối chuẩn cùng quy mô ~1.93M)')
print(f'3. DarkIR-m gốc (C=32, tham chiếu):   {n_params(darkir_m_ref):,} tham số (khớp chuẩn 3,321,638 của CVPR 2025)')

x_test = torch.rand(1, 3, 61, 83)
for name, m in [('DarkIR-Edge', edge), ('Standard U-Net', unet)]:
    out, side = m(x_test, return_side=True)
    out.mean().backward()
    print(f'  [OK] {name}: input {tuple(x_test.shape)} -> output {tuple(out.shape)} | side {None if side is None else tuple(side.shape)}')
assert out.shape == x_test.shape
del edge, darkir_m_ref, unet

In [ ]:
# ===== CELL 11: Hàm mất mát tổng hợp CompositeLoss (gồm cả Cosine Color Loss), EMA & Metrics =====
def _gauss_win(size=11, sigma=1.5, ch=3):
    g = torch.arange(size).float() - size // 2
    g = torch.exp(-g ** 2 / (2 * sigma ** 2))
    g /= g.sum()
    return (g[:, None] * g[None, :]).expand(ch, 1, size, size).contiguous()

def ssim_torch(x, y, win):
    ch = x.shape[1]; w = win.to(x.device)
    mu_x, mu_y = F.conv2d(x, w, groups=ch), F.conv2d(y, w, groups=ch)
    sxx = F.conv2d(x * x, w, groups=ch) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=ch) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=ch) - mu_x * mu_y
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu_x * mu_y + C1) * (2 * sxy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sxx + syy + C2))).mean()

class CompositeLoss(nn.Module):
    '''
    Hàm mất mát đa nhiệm chuẩn Tài liệu 03 (Phần IV.1):
    L_total = L1(pred, gt)
            + w_side  * L1(side, avg_pool2d(gt, 8))             [Deep Supervision tại Bottleneck]
            + w_ssim  * (1 - SSIM(pred, gt))                    [Bảo toàn cấu trúc bề mặt]
            + w_color * CosineColorLoss(pred, gt, mask>=5/255)  [Chống lệch màu theo tinh thần HVI/Retinex]
    '''
    def __init__(self, w, use_color=True):
        super().__init__()
        self.w, self.use_color = w, use_color
        self.win = _gauss_win()

    def forward(self, pred, gt, side=None):
        pred, gt = pred.float(), gt.float()
        parts = {'l1': F.l1_loss(pred, gt)}
        parts['ssim'] = self.w['ssim'] * (1.0 - ssim_torch(pred.clamp(0, 1), gt, self.win))
        if side is not None:
            gt_down = F.avg_pool2d(gt, kernel_size=8, stride=8)
            parts['side'] = self.w['side'] * F.l1_loss(side.float(), gt_down)
        if self.use_color:
            # Cosine Color Loss kèm mặt nạ che các pixel tối sâu (< 5/255) để chống bùng nổ gradient góc màu
            cos = F.cosine_similarity(pred, gt, dim=1, eps=1e-6)
            mask = (gt.amax(1) >= 5.0 / 255.0).float()
            parts['color'] = self.w['color'] * (1.0 - (cos * mask).sum() / mask.sum().clamp(min=1.0))
        return sum(parts.values()), parts

class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval()
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for pe, p in zip(self.m.parameters(), model.parameters()):
            pe.mul_(d).add_(p.detach(), alpha=1 - d)
        for be, b in zip(self.m.buffers(), model.buffers()):
            be.copy_(b)

def psnr_u8(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 99.0 if mse == 0 else 10.0 * math.log10(255.0 ** 2 / mse)

@torch.no_grad()
def predict_u8(model, img_u8):
    x = to_tensor(img_u8).unsqueeze(0).to(DEV)
    with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
        y = model(x)
    return (y.float().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

@torch.no_grad()
def val_psnr(model, ds):
    model.eval()
    return float(np.mean([psnr_u8(predict_u8(model, lo), hi) for lo, hi in ds.cache]))

## 🚀 Huấn luyện Mô hình (Tách riêng Cell cho `DarkIR-Edge` và Cell cho `Standard U-Net`)
- **Về cơ chế `EarlyStopping` trong `train_model` (Cell 12):**
  - Toàn bộ khối kiểm tra `EarlyStopping` được đóng khung rõ ràng bằng chú thích `=== [KHỐI EARLY STOPPING] ===` ngay bên trong `train_model`.
  - **Trạng thái mặc định hiện tại:** Đang **TẮT (commented / `CFG['use_early_stopping'] = False`)** để mô hình chạy trọn vẹn 100 epochs (30.000 steps) cho lịch trình `CosineAnnealingLR` hội tụ mượt về $10^{-6}$ và luôn lưu thêm `best.pt` song song với `final.pt`. Nếu bạn muốn bật dừng sớm theo ý đồ riêng, chỉ cần đổi `CFG['use_early_stopping'] = True` (hoặc bỏ comment 4 dòng trong khối đó).
- **Thứ tự ưu tiên chạy:**
  - **Cell 13:** Đo tốc độ thực tế (`probe_speed`) 200 bước đầu để biết chính xác số phút/run trên GPU Colab hiện tại.
  - **Cell 14 (ƯU TIÊN SỐ 1):** Chạy huấn luyện riêng cho mô hình chính **`DarkIR-Edge (C=24)`**.
  - **Cell 15 (TÙY CHỌN):** Chạy huấn luyện riêng cho **`Standard U-Net`** đối chuẩn.

In [ ]:
# ===== CELL 12: Hàm huấn luyện cốt lõi (AMP + Gradient Clipping + EMA + Auto-Resume + Khối Early Stopping rõ ràng) =====
def build(tag):
    if tag == 'darkir_edge': return DarkIREdge(width=24)
    if tag == 'unet':        return UNetBaseline(CFG['unet_base'])
    raise ValueError(f'Không nhận diện được mô hình: {tag}')

def train_model(tag, seed):
    out_dir = Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Kiểm tra trạng thái huấn luyện cũ & cờ FORCE_RETRAIN
    if (out_dir / 'final.pt').exists() and not CFG.get('force_retrain', False):
        print(f'[SKIP] {tag} (seed={seed}) trong phiên bản "{CFG["version"]}" đã có final.pt -> bỏ qua.')
        print(f'       (Để train lại, hãy đổi EXP_VERSION sang version mới hoặc đặt FORCE_RETRAIN=True).')
        return
    elif (out_dir / 'final.pt').exists() and CFG.get('force_retrain', False):
        print(f'[RETRAIN] Phát hiện final.pt cũ nhưng FORCE_RETRAIN=True -> Huấn luyện lại từ đầu cho {tag} (seed={seed})!')

    # Khóa cứng hạt giống trước khi khởi tạo mô hình và DataLoader -> tái lập 100%
    set_seed(seed)
    model = build(tag).to(DEV)
    ema = EMA(model, CFG['ema_decay'])
    crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], betas=CFG['betas'], weight_decay=CFG['weight_decay'])
    loader = make_train_loader(seed)
    spe = len(loader) if CFG['max_steps_per_epoch'] is None else min(len(loader), CFG['max_steps_per_epoch'])
    total_steps = CFG['epochs'] * spe
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=CFG['lr_min'])
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    history, start_ep = [], 0
    best_val_psnr = -1.0
    early_stop_wait = 0

    # Auto-Resume: chỉ resume nếu có last.pt và không bật FORCE_RETRAIN
    if (out_dir / 'last.pt').exists() and not CFG.get('force_retrain', False):
        ck = torch.load(out_dir / 'last.pt', map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); ema.m.load_state_dict(ck['ema'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        history, start_ep, ema.n = ck['history'], ck['epoch'], ck['ema_n']
        best_val_psnr = ck.get('best_val_psnr', -1.0)
        early_stop_wait = ck.get('early_stop_wait', 0)
        print(f'[RESUME] Khôi phục {tag} (seed={seed}) từ epoch {start_ep}/{CFG["epochs"]}')

    print(f'=== Bắt đầu huấn luyện {tag} (seed={seed}) [Version: {CFG["version"]}] | {n_params(model):,} params | {spe} steps/ep × {CFG["epochs"]} ep = {total_steps:,} steps ===')
    t0 = time.time()

    for ep in range(start_ep, CFG['epochs']):
        model.train()
        run_metrics, n_batches = {}, 0
        for it, (lo, hi) in enumerate(loader):
            if it >= spe: break
            lo, hi = lo.to(DEV, non_blocking=True), hi.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
                pred, side = model(lo, return_side=True)
            loss, parts = crit(pred, hi, side)
            if not torch.isfinite(loss):
                print(f'WARN: Loss không hữu hạn tại ep {ep+1} step {it} -> bỏ qua bước này')
                opt.zero_grad(set_to_none=True); continue

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(model)

            run_metrics['loss'] = run_metrics.get('loss', 0.0) + loss.item()
            for k, v in parts.items():
                run_metrics[k] = run_metrics.get(k, 0.0) + v.item()
            n_batches += 1

        rec = {'epoch': ep + 1, 'lr': sched.get_last_lr()[0], **{k: v / max(n_batches, 1) for k, v in run_metrics.items()}}
        if (ep + 1) % CFG['val_interval'] == 0 or (ep + 1) == CFG['epochs']:
            cur_val = val_psnr(ema.m, val_ds)
            rec['val_psnr_ema'] = cur_val

            # Luôn lưu lại best.pt khi đạt đỉnh Val PSNR mới để bạn tùy ý sử dụng nếu cần
            if cur_val > best_val_psnr:
                best_val_psnr = cur_val
                early_stop_wait = 0
                torch.save({'ema': ema.m.state_dict(), 'model': model.state_dict(), 'epoch': ep + 1, 'val_psnr': cur_val}, out_dir / 'best.pt')
            else:
                early_stop_wait += CFG['val_interval']

            torch.save({
                'model': model.state_dict(), 'ema': ema.m.state_dict(),
                'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'history': history + [rec], 'epoch': ep + 1, 'ema_n': ema.n,
                'best_val_psnr': best_val_psnr, 'early_stop_wait': early_stop_wait
            }, out_dir / 'last.pt')

            # =====================================================================
            # === [KHỐI EARLY STOPPING — HIỆN ĐANG TẠM KHÓA THEO YÊU CẦU CỦA BẠN] ===
            # Mặc định chúng ta chạy đủ 100 epochs cố định để CosineAnnealingLR hội tụ
            # hết chu kỳ. Nếu bạn muốn kích hoạt dừng sớm, hãy bỏ comment 4 dòng dưới:
            # =====================================================================
            # if CFG.get('use_early_stopping', False) and early_stop_wait >= CFG.get('early_stop_patience', 20):
            #     print(f'[EARLY STOPPING] Dừng sớm tại epoch {ep+1} do Val PSNR không tăng sau {early_stop_wait} epochs (Best={best_val_psnr:.2f} dB).')
            #     history.append(rec)
            #     break
            # =====================================================================

        history.append(rec)
        if 'val_psnr_ema' in rec or ep == start_ep:
            elapsed = (time.time() - t0) / 60.0
            print(f'Epoch {ep+1:3d}/{CFG["epochs"]} | Loss: {rec.get("loss", float("nan")):.4f} | LR: {rec["lr"]:.2e} | Val PSNR (EMA): {rec.get("val_psnr_ema", float("nan")):.2f} dB (Best: {best_val_psnr:.2f}) | {elapsed:.1f} phút')

    pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
    torch.save({
        'ema': ema.m.state_dict(), 'model': model.state_dict(),
        'tag': tag, 'seed': seed, 'best_val_psnr': best_val_psnr,
        'cfg': {k: v for k, v in CFG.items() if k != 'runs'}
    }, out_dir / 'final.pt')
    print(f'✔ Hoàn tất {tag} (seed={seed}) trong {(time.time() - t0) / 60.0:.1f} phút -> Đã lưu final.pt & best.pt tại {out_dir}')

In [ ]:
# ===== CELL 13: ĐO TỐC ĐỘ THỰC TẾ (200 bước đầu) để ước lượng chính xác số phút mỗi lần chạy trên Colab =====
def probe_speed(tag, steps=200):
    set_seed(MASTER_SEED)
    m = build(tag).to(DEV).train()
    crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    loader = make_train_loader(MASTER_SEED)
    it_loader = iter(loader)
    steps = 6 if TEST_MODE else steps
    t_start = None
    for i in range(steps):
        if i == 3:
            if DEV.type == 'cuda': torch.cuda.synchronize()
            t_start = time.time()
        try:
            lo, hi = next(it_loader)
        except StopIteration:
            it_loader = iter(loader); lo, hi = next(it_loader)
        lo, hi = lo.to(DEV), hi.to(DEV)
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
            pred, side = m(lo, return_side=True)
        loss, _ = crit(pred, hi, side)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    if DEV.type == 'cuda': torch.cuda.synchronize()
    ips = (steps - 3) / max(time.time() - t_start, 1e-6)
    total_steps = CFG['epochs'] * (len(train_ds) // CFG['batch_size'])
    est_min = total_steps / ips / 60.0
    print(f'[{tag:12s}] Tốc độ thực tế: {ips:.2f} it/s -> Ước tính 1 run ({total_steps:,} steps): ~{est_min:.1f} phút')
    del m, crit, opt, scaler

probe_speed('darkir_edge')
probe_speed('unet')

In [ ]:
# ===== CELL 14 (ƯU TIÊN SỐ 1): HUẤN LUYỆN MÔ HÌNH ĐỀ XUẤT CHÍNH — DarkIR-Edge (C=24) =====
# Mặc định chạy các seed trong CFG['darkir_edge_seeds'] = [42, 123].
# Nếu bạn muốn chạy trước 1 seed chính duy nhất để lấy kết quả nhanh nhất, chỉ cần đổi thành:
# SEEDS_TO_RUN_DARKIR_EDGE = [42]

SEEDS_TO_RUN_DARKIR_EDGE = CFG['darkir_edge_seeds']   # Mặc định: [42, 123]

for seed in SEEDS_TO_RUN_DARKIR_EDGE:
    train_model('darkir_edge', seed=seed)

In [ ]:
# ===== CELL 15 (TÁCH RIÊNG - TÙY CHỌN): HUẤN LUYỆN BASELINE ĐỐI CHUẨN — Standard U-Net =====
# Đặt RUN_UNET_BASELINE = True nếu bạn muốn huấn luyện thêm Standard U-Net (~1.93M params) để so sánh đối chuẩn.
# Đặt RUN_UNET_BASELINE = False nếu bạn chỉ muốn tập trung chạy DarkIR-Edge trước.

RUN_UNET_BASELINE = True
SEEDS_TO_RUN_UNET = CFG['unet_seeds']                 # Mặc định: [42]

if RUN_UNET_BASELINE:
    for seed in SEEDS_TO_RUN_UNET:
        train_model('unet', seed=seed)
else:
    print('[SKIP] Bạn đang tắt RUN_UNET_BASELINE -> Bỏ qua huấn luyện Standard U-Net.')

## 📊 Đánh giá Toàn diện trên 150 Ảnh Test & Kiểm định Thống kê
- Tự động quét tất cả các checkpoint `final.pt` đã huấn luyện xong trong `CFG['ckpt_dir']` (dù bạn chỉ mới chạy xong 1 seed của `DarkIR-Edge`, hay đã chạy đủ cả `DarkIR-Edge` và `Standard U-Net`, các cell đánh giá bên dưới đều tự động thích ứng mà không báo lỗi).

In [ ]:
# ===== CELL 16: Baselines cổ điển (Gamma, CLAHE, MSRCR) & Hàm tính Metrics chuẩn =====
from skimage.metrics import structural_similarity

def gamma_fn(img, g=2.2):
    return (np.power(img / 255.0, 1.0 / g) * 255.0).round().astype(np.uint8)

def clahe_fn(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def msrcr_fn(img, sigmas=(15, 80, 250), alpha=125., beta=46., G=192., off=-30., lo_p=1, hi_p=99):
    x = img.astype(np.float64) + 1.0
    msr = sum(np.log10(x) - np.log10(cv2.GaussianBlur(x, (0, 0), s)) for s in sigmas) / len(sigmas)
    cr = beta * (np.log10(alpha * x) - np.log10(x.sum(axis=2, keepdims=True)))
    out = G * (msr * cr - off)
    for c in range(3):
        lo, hi = np.percentile(out[..., c], [lo_p, hi_p])
        out[..., c] = np.clip((out[..., c] - lo) / (hi - lo + 1e-6), 0, 1) * 255.0
    return out.round().astype(np.uint8)

LP = None
if LPIPS_OK:
    try:
        LP = lpips.LPIPS(net='alex', verbose=False).to(DEV).eval()
    except Exception as e:
        print('WARN: Không tải được trọng số AlexNet cho LPIPS -> bỏ qua LPIPS.', e)
        LPIPS_OK = False

@torch.no_grad()
def metrics(pred, gt):
    r = {
        'psnr': psnr_u8(pred, gt),
        'ssim': float(structural_similarity(pred, gt, channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))
    }
    scale = gt.mean() / max(pred.mean(), 1e-6)
    pred_norm = np.clip(pred.astype(np.float64) * scale, 0, 255).round().astype(np.uint8)
    r['psnr_gtmean'] = psnr_u8(pred_norm, gt)   # Chỉ số chẩn đoán phân rã lỗi nội bộ
    if LP is not None:
        t = lambda a: (to_tensor(a).unsqueeze(0) * 2.0 - 1.0).to(DEV)
        r['lpips'] = float(LP(t(pred), t(gt)).item())
    return r

def eval_method(name, fn, seed=None):
    rows = []
    for domain, pairs in SPLITS['test'].items():
        for lp, hp in pairs:
            lo, hi = imread_rgb(lp), imread_rgb(hp)
            rows.append({'method': name, 'seed': seed, 'domain': domain, 'image': Path(lp).name, **metrics(fn(lo), hi)})
    return pd.DataFrame(rows)

def find_checkpoint(tag, seed, ckpt_name='final.pt'):
    # Chỉ tìm trong đúng version hiện tại (experiments/{version}/checkpoints/) -> không trộn checkpoint giữa các version
    for name in (ckpt_name, 'best.pt'):
        c = Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}' / name
        if c.exists():
            return c
    return None

def nn_fn(tag, seed, ckpt_name='final.pt'):
    ckpt_path = find_checkpoint(tag, seed, ckpt_name)
    if ckpt_path is None:
        raise FileNotFoundError(f"Không tìm thấy checkpoint cho {tag} (seed={seed}) trong {CFG['ckpt_dir']}.")
    ck = torch.load(ckpt_path, map_location=DEV, weights_only=False)
    m = build(tag).to(DEV).eval()
    m.load_state_dict(ck['ema'])
    return lambda img: predict_u8(m, img)

In [ ]:
# ===== CELL 17: Chạy đánh giá trên toàn bộ 150 ảnh test -> Lưu results_per_image.csv =====
frames = [
    eval_method('Input (Low-Light)', lambda x: x),
    eval_method('Gamma 2.2', gamma_fn),
    eval_method('CLAHE', clahe_fn),
    eval_method('MSRCR (Retinex)', msrcr_fn),
]
NAME_MAP = {'darkir_edge': 'DarkIR-Edge (ours)', 'unet': 'Standard U-Net'}
ALL_CANDIDATE_RUNS = [('darkir_edge', CFG['darkir_edge_seeds']), ('unet', CFG['unet_seeds'])]

for tag, seeds in ALL_CANDIDATE_RUNS:
    for s in seeds:
        ckpt_file = find_checkpoint(tag, s)
        if ckpt_file is not None:
            print(f'Đang đánh giá {NAME_MAP[tag]} (seed={s}) từ {ckpt_file.name} tại: {ckpt_file.parent.name}...')
            frames.append(eval_method(NAME_MAP[tag], nn_fn(tag, s, ckpt_name=ckpt_file.name), seed=s))
        else:
            print(f'WARN: Chưa có checkpoint cho {tag} seed {s} trong {CFG["ckpt_dir"]}')

RES = pd.concat(frames, ignore_index=True)
res_csv_path = f"{CFG['result_dir']}/results_per_image.csv"
RES.to_csv(res_csv_path, index=False)
print(f'✔ Đã lưu chi tiết 150 ảnh test vào: {res_csv_path}')

# Đồng bộ một bản sao vào thư mục kết quả mới nhất results_latest
if os.path.exists(CFG['latest_dir']):
    RES.to_csv(f"{CFG['latest_dir']}/results_per_image.csv", index=False)

cols_show = ['psnr', 'ssim', 'psnr_gtmean'] + (['lpips'] if LPIPS_OK and 'lpips' in RES.columns else [])
print(RES.groupby(['method', 'seed'], dropna=False)[cols_show].mean().round(3))

In [ ]:
# ===== CELL 18: Tổng hợp Bảng kết quả chính (Mean ± Std qua seed) & Kiểm định ghép cặp (Paired Tests) =====
from scipy import stats
METRICS = [m for m in ['psnr', 'ssim', 'lpips'] if m in RES.columns]

def summarize(df):
    rows = []
    for (meth, seed), g in df.groupby(['method', 'seed'], dropna=False):
        r = {'method': meth, 'seed': seed}
        for m in METRICS:
            dom = g.groupby('domain')[m].mean()
            for d, v in dom.items(): r[f'{m}|{d}'] = v
            r[f'{m}|pooled'] = g[m].mean()
            r[f'{m}|macro']  = dom.mean()
        rows.append(r)
    per_seed = pd.DataFrame(rows)
    agg = per_seed.drop(columns='seed').groupby('method', sort=False).agg(['mean', 'std'])
    return per_seed, agg

per_seed, agg = summarize(RES)
def fmt(meth, col):
    mu, sd = agg.loc[meth, (col, 'mean')], agg.loc[meth, (col, 'std')]
    return f'{mu:.3f}' + (f' ± {sd:.3f}' if not np.isnan(sd) else '')

cols = [c for c in per_seed.columns if c not in ('method', 'seed')]
TABLE = pd.DataFrame({c: [fmt(m, c) for m in agg.index] for c in cols}, index=agg.index)

# Lưu bảng kết quả chính vào folder version trên Drive
table_path = f"{CFG['result_dir']}/table_main.csv"
TABLE.to_csv(table_path)
print(f'✔ Đã lưu Bảng kết quả chính vào: {table_path}')

# Đồng bộ bản sao sang results_latest
if os.path.exists(CFG['latest_dir']):
    TABLE.to_csv(f"{CFG['latest_dir']}/table_main.csv")

pd.set_option('display.width', 260, 'display.max_columns', 50)
print('=== BẢNG PSNR (dB, cao hơn là tốt hơn) ===')
print(TABLE[[c for c in cols if c.startswith('psnr|')]])
print()
print('=== BẢNG SSIM (cao hơn là tốt hơn) ===')
print(TABLE[[c for c in cols if c.startswith('ssim|')]])
if LPIPS_OK and any(c.startswith('lpips|') for c in cols):
    print()
    print('=== BẢNG LPIPS (thấp hơn là tốt hơn) ===')
    print(TABLE[[c for c in cols if c.startswith('lpips|')]])

# --- Bảng chẩn đoán phân rã lỗi GT-mean (Diagnostic Table) ---
diag = RES.groupby(['method', 'domain'])[['psnr', 'psnr_gtmean']].mean()
diag['gap_illumination_dB'] = diag['psnr_gtmean'] - diag['psnr']
diag_path = f"{CFG['result_dir']}/diagnostic_gtmean.csv"
diag.to_csv(diag_path)
if os.path.exists(CFG['latest_dir']):
    diag.to_csv(f"{CFG['latest_dir']}/diagnostic_gtmean.csv")
print()
print('=== CHẨN ĐOÁN NỘI BỘ: ĐỘ LỆCH ÁNH SÁNG TOÀN CỤC (psnr_gtmean - psnr) ===')
print(diag.round(2))

# --- Kiểm định thống kê ghép cặp từng ảnh (Paired t-test, Wilcoxon & Bootstrap 95% CI) ---
A = 'DarkIR-Edge (ours)'
B = 'Standard U-Net' if 'Standard U-Net' in RES.method.values else 'CLAHE'
if A in RES.method.values and B in RES.method.values:
    pa = RES[RES.method == A].groupby(['domain', 'image']).psnr.mean()
    pb = RES[RES.method == B].groupby(['domain', 'image']).psnr.mean()
    d = (pa - pb).dropna().values
    rng = np.random.default_rng(MASTER_SEED)
    ci = np.percentile([rng.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5])
    t_pval = stats.ttest_rel(pa.loc[pb.index], pb).pvalue
    w_pval = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    print()
    print(f'[Kiểm định Ghép cặp trên N={len(d)} ảnh test]: ΔPSNR ({A} − {B})')
    print(f'  Mean ΔPSNR = {d.mean():+.3f} dB | 95% Bootstrap CI: [{ci[0]:+.3f}, {ci[1]:+.3f}] dB')
    print(f'  Paired t-test p = {t_pval:.4g} | Wilcoxon p = {w_pval:.4g} | Tỷ lệ ảnh {A} > {B}: {(d > 0).mean():.1%}')

# Liệt kê tất cả các phiên bản thực nghiệm đã có trong Google Drive:
exp_base = Path(DRIVE) / 'experiments'
if exp_base.exists():
    all_runs = sorted([d.name for d in exp_base.iterdir() if d.is_dir()])
    print()
    print(f"📁 Danh sách các phiên bản thực nghiệm đã lưu trong Google Drive ({len(all_runs)} runs): {all_runs}")

In [ ]:
# ===== CELL 19: Xuất lưới ảnh định tính (Qualitative Grid), Đường cong học tập & Ảnh không cặp (DICM/LIME) =====
import matplotlib.pyplot as plt

def pick_examples(k=2):
    out = []
    for d, pairs in SPLITS['test'].items():
        out += [(d, p) for p in pairs[:k]]
    return out

ex = pick_examples(2)
fns = {'Input': lambda x: x, 'CLAHE': clahe_fn}
for s in CFG['unet_seeds']:
    ckpt_file = find_checkpoint('unet', s)
    if ckpt_file is not None:
        fns['Standard U-Net'] = nn_fn('unet', s, ckpt_name=ckpt_file.name)
        break

for s in CFG['darkir_edge_seeds']:
    ckpt_file = find_checkpoint('darkir_edge', s)
    if ckpt_file is not None:
        fns['DarkIR-Edge'] = nn_fn('darkir_edge', s, ckpt_name=ckpt_file.name)
        break

fig, ax = plt.subplots(len(ex), len(fns) + 1, figsize=(3.2 * (len(fns) + 1), 2.5 * len(ex)))
ax = np.atleast_2d(ax)
for r, (dom, (lp, hp)) in enumerate(ex):
    lo, hi = imread_rgb(lp), imread_rgb(hp)
    for c, (name, fn) in enumerate(fns.items()):
        im = fn(lo)
        ax[r, c].imshow(im)
        title = f'{name} ({psnr_u8(im, hi):.2f} dB)' if name != 'Input' else f'Input ({dom}, {psnr_u8(lo, hi):.2f} dB)'
        ax[r, c].set_title(title, fontsize=8.5)
        ax[r, c].axis('off')
    ax[r, -1].imshow(hi)
    ax[r, -1].set_title('Ground Truth', fontsize=8.5)
    ax[r, -1].axis('off')
plt.tight_layout()
qual_path = f"{CFG['result_dir']}/qualitative.png"
plt.savefig(qual_path, dpi=150)
if os.path.exists(CFG['latest_dir']):
    plt.savefig(f"{CFG['latest_dir']}/qualitative.png", dpi=150)
plt.show()
print(f'✔ Đã lưu Lưới ảnh so sánh định tính vào: {qual_path}')

# --- Tự động lưu từng ảnh phục chế mẫu kích thước gốc vào folder visual_samples trên Drive ---
vis_dir = Path(CFG['result_dir']) / 'visual_samples'
vis_dir.mkdir(parents=True, exist_ok=True)
for r, (dom, (lp, hp)) in enumerate(ex):
    base_name = Path(lp).stem
    cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_input.png'), cv2.cvtColor(imread_rgb(lp), cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_gt.png'), cv2.cvtColor(imread_rgb(hp), cv2.COLOR_RGB2BGR))
    for name, fn in fns.items():
        if name != 'Input':
            san_name = name.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
            cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_{san_name}.png'), cv2.cvtColor(fn(imread_rgb(lp)), cv2.COLOR_RGB2BGR))
print(f'✔ Đã lưu các ảnh phục chế mẫu độ phân giải cao vào: {vis_dir}')

# --- Vẽ đường cong huấn luyện (Loss & Validation PSNR) ---
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.5))
has_history = False
for tag, seeds in [('darkir_edge', CFG['darkir_edge_seeds']), ('unet', CFG['unet_seeds'])]:
    for s in seeds:
        h = Path(CFG['ckpt_dir']) / f'{tag}_seed{s}' / 'history.csv'
        if h.exists():
            has_history = True
            df = pd.read_csv(h)
            ax[0].plot(df.epoch, df.loss, label=f'{tag} (s={s})')
            v = df.dropna(subset=['val_psnr_ema'])
            ax[1].plot(v.epoch, v.val_psnr_ema, marker='o', markersize=3, label=f'{tag} (s={s})')

if has_history:
    ax[0].set_title('Training Composite Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].set_title('Validation PSNR (EMA, dB)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    curve_path = f"{CFG['result_dir']}/curves.png"
    plt.savefig(curve_path, dpi=150)
    if os.path.exists(CFG['latest_dir']):
        plt.savefig(f"{CFG['latest_dir']}/curves.png", dpi=150)
    plt.show()
    print(f'✔ Đã lưu Biểu đồ đường cong học tập vào: {curve_path}')
else:
    plt.close()

In [ ]:
# ===== CELL 20: Bảng Hiệu năng Phần cứng (Params, GFLOPs, Latency FP32/FP16, FPS, VRAM) =====
from torch.utils.flop_counter import FlopCounterMode

rows = []
for tag, name in [('darkir_edge', 'DarkIR-Edge (C=24)'), ('unet', 'Standard U-Net (base=32)')]:
    m = build(tag).to(DEV).eval()
    r = {'Model': name, 'Params (M)': round(n_params(m) / 1e6, 3), 'Exact Params': n_params(m)}
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.rand(1, 3, 256, 256, device=DEV))
    r['GFLOPs @ 256x256 (Spatial)'] = round(fc.get_total_flops() / 1e9, 2)
    if DEV.type == 'cuda':
        x_bench = torch.rand(1, 3, 400, 600, device=DEV)
        for prec, dt in [('FP32', None), ('FP16', torch.float16)]:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=dt is not None):
                for _ in range(10): m(x_bench)
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
                t0 = time.time()
                for _ in range(50): m(x_bench)
                torch.cuda.synchronize()
                ms = (time.time() - t0) / 50.0 * 1000.0
            r[f'Latency {prec} @600x400 (ms)'] = round(ms, 2)
            r[f'FPS {prec} @600x400'] = round(1000.0 / ms, 1)
        r['Peak VRAM (MB)'] = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1)
    rows.append(r)

EFF = pd.DataFrame(rows)
eff_path = f"{CFG['result_dir']}/efficiency.csv"
EFF.to_csv(eff_path, index=False)
if os.path.exists(CFG['latest_dir']):
    EFF.to_csv(f"{CFG['latest_dir']}/efficiency.csv", index=False)
print(f'✔ Đã lưu Bảng hiệu năng phần cứng vào: {eff_path}')
print(EFF.T)

## (Tùy chọn) Đánh giá Mốc Trần Tham Chiếu `DarkIR-m` Chính Thức (CVPR 2025) trên 150 Ảnh Test
Vì lớp `DarkIREdge(width=32, use_imax=False, use_gate=False)` ở Cell 8 khớp 1:1 tên biến và cấu trúc của `DarkIR/archs/DarkIR.py`, bạn có thể bật `RUN_OFFICIAL_DARKIR_REF = True` ở Cell 22 dưới đây để tự động tải trọng số chính thức của tác giả từ HuggingFace (`Cidaut/DarkIR`) và đánh giá làm mốc trần tham chiếu.

In [ ]:
# ===== CELL 22 (Tùy chọn): Tự động tải & chạy Checkpoint DarkIR-m (C=32) chính thức từ HuggingFace =====
RUN_OFFICIAL_DARKIR_REF = False   # Đổi thành True nếu muốn chạy thêm mốc trần tham chiếu DarkIR-m (CVPR 2025)

if RUN_OFFICIAL_DARKIR_REF:
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
        from huggingface_hub import hf_hub_download, list_repo_files

    files = [f for f in list_repo_files('Cidaut/DarkIR') if f.endswith('.pt') or f.endswith('.pth')]
    print('Các checkpoint có trong HuggingFace Cidaut/DarkIR:', files)
    target_ckpt = next((f for f in files if '1k' in f or '384' in f or 'lol' in f.lower()), files[0] if files else None)
    if target_ckpt:
        ckpt_path = hf_hub_download(repo_id='Cidaut/DarkIR', filename=target_ckpt)
        raw = torch.load(ckpt_path, map_location=DEV, weights_only=False)
        state = raw.get('params', raw.get('model_state_dict', raw))
        clean_state = {}
        for k, v in state.items():
            k2 = k.replace('module.', '').replace('.modules_list.', '.')
            if 'adapter' not in k2:
                clean_state[k2] = v
        ref_model = DarkIREdge(width=32, use_imax=False, use_gate=False).to(DEV).eval()
        missing, unexpected = ref_model.load_state_dict(clean_state, strict=False)
        print(f'Đã nạp {target_ckpt} | missing={len(missing)}, unexpected={len(unexpected)}')
        ref_df = eval_method('DarkIR-m (Official CVPR25 Ref)', lambda im: predict_u8(ref_model, im), seed=0)
        RES_WITH_REF = pd.concat([RES, ref_df], ignore_index=True)
        ref_path = f"{CFG['result_dir']}/results_per_image_with_ref.csv"
        RES_WITH_REF.to_csv(ref_path, index=False)
        if os.path.exists(CFG['latest_dir']):
            RES_WITH_REF.to_csv(f"{CFG['latest_dir']}/results_per_image_with_ref.csv", index=False)
        print(f'✔ Đã lưu kết quả mốc trần tham chiếu vào: {ref_path}')
        print(RES_WITH_REF.groupby('method')[['psnr', 'ssim']].mean().round(3))
